> ⏸ **Parked: not part of the current phase.** This notebook will be updated once model training and evaluation (notebooks 01–03) are finalised. Don't run it yet.

# 07 · Faithfulness and hallucination evaluation

The project's main experiment: over `N` test images, compares ungrounded vs. grounded explanations on heatmap–explanation IoU, the pointing game and hallucination rates, with paired significance tests.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 (not P100)
- **Internet:** On
- **Inputs:** output of **01_extract_faces**; output of **02_train**
- **Secrets (Add-ons → Secrets):** one of `ANTHROPIC_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY` or `LLM_API_KEY` (none needed for Ollama)

**Produces:** `/kaggle/working/outputs/eval/` (per-image CSV, summary, plots). When it finishes, use **Save Version → Save & Run All**, then add this notebook's output as an input to the next notebook.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup, setup_llm, start_ollama
setup(data=True, checkpoint=True)

# --- LLM backend -------------------------------------------------------------------------
LLM_PROVIDER = "anthropic"   # anthropic | openrouter | ollama | openai (any OpenAI-compatible server)
LLM_MODEL = None             # None = provider default; e.g. "meta-llama/llama-3.3-70b-instruct" on OpenRouter
if LLM_PROVIDER == "ollama":
    start_ollama(LLM_MODEL or "llama3.1:8b")   # local model running inside this Kaggle session
assert setup_llm(LLM_PROVIDER, LLM_MODEL), "Add your API key under Add-ons → Secrets"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

## Run
`N` images × 2 conditions = `2N` API calls. Add `--include-real` to also sample real frames.

In [ ]:
N = 200
!python -m src.eval.run_eval --n {N}

In [ ]:
res = pd.read_csv("/kaggle/working/outputs/eval/results.csv")
ok = res[res.error.isna()] if "error" in res else res
metrics = ["iou", "pointing_hit", "region_unsupported", "artifact_unsupported", "verdict_mismatch",
           "citation_invalid", "any_hallucination"]
ok = ok.copy(); ok[metrics] = ok[metrics].astype(float)
summary = ok.groupby("condition")[metrics].mean().T[["ungrounded", "grounded"]]
display(summary.round(3))
print("errors per condition:", res.groupby("condition").error.count().to_dict() if "error" in res else 0)

In [ ]:
summary.drop(index="iou").plot.barh(figsize=(8, 4), title="Rates by condition (lower is better, except pointing_hit)")
plt.xlim(0, 1); plt.tight_layout(); plt.savefig("/kaggle/working/outputs/eval/rates.png", dpi=150); plt.show()

## Paired significance tests
Same images in both conditions → McNemar's exact test for each binary flag and Wilcoxon signed-rank for IoU.

In [ ]:
from scipy.stats import binomtest, wilcoxon

paired = ok.pivot_table(index="image", columns="condition", values=metrics, aggfunc="first").dropna()
rows = []
for mt in metrics:
    u, g = paired[(mt, "ungrounded")].astype(float), paired[(mt, "grounded")].astype(float)
    if mt == "iou":
        p = wilcoxon(u, g).pvalue if (u != g).any() else 1.0
        rows.append({"metric": mt, "ungrounded": u.mean(), "grounded": g.mean(), "test": "Wilcoxon", "p": p})
    else:
        b, c = int(((u == 1) & (g == 0)).sum()), int(((u == 0) & (g == 1)).sum())
        p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
        rows.append({"metric": mt, "ungrounded": u.mean(), "grounded": g.mean(), "test": f"McNemar ({b} vs {c})", "p": p})
tests = pd.DataFrame(rows).set_index("metric")
tests.to_csv("/kaggle/working/outputs/eval/significance.csv")
display(tests.round(4))
print(f"n = {len(paired)} paired images")

## By manipulation method

In [ ]:
ok = ok.assign(method=ok.image.str.split("__").str[0])
by = ok.pivot_table(index="method", columns="condition", values=["any_hallucination", "iou"], aggfunc="mean")
display(by.round(3))
by["any_hallucination"][["ungrounded", "grounded"]].plot.bar(rot=0, figsize=(8, 3.5), ylim=(0, 1),
                                                             title="Hallucination rate by method"); plt.show()